# SecurePay AI - Local Evaluation on AMD AI Notebooks
This notebook was designed for the **AMD Developer Cloud (ROCm 7.2 + vLLM)** environment. We utilized the AMD Notebooks during the hackathon to evaluate inference latencies, prototype our Chain-of-Thought risk scoring prompts, and ensure compatibility with Google Gemma 3 on AMD MI300X accelerators before moving to the Fireworks Serverless API for our production frontend.

### 🔑 Hugging Face Authentication (Optional)
Google's Gemma models are gated on Hugging Face. To load the Gemma 3 model, make sure you have accepted the terms on the [Hugging Face model page](https://huggingface.co/google/gemma-3-4b-it).

If you have a Hugging Face token, you can run `huggingface-cli login` in the terminal or uncomment the cell below to authenticate. If no token is provided, the notebook will automatically fall back to the open, non-gated **Qwen 2.5 7B Instruct** model so you can evaluate the ROCm+vLLM pipeline without interruptions.

In [ ]:
# # Optional: Login to Hugging Face to load gated Gemma models
# from huggingface_hub import login
# login("your_hugging_face_token_here")

In [ ]:
import torch
from vllm import LLM, SamplingParams

print(f"PyTorch version: {torch.__version__}")
print(f"ROCm available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

In [ ]:
# Initialize vLLM engine optimized for AMD MI300X
try:
    print("Attempting to load Google Gemma 3 model (google/gemma-3-4b-it)...\n")
    llm = LLM(model="google/gemma-3-4b-it", trust_remote_code=True, tensor_parallel_size=1)
except Exception as e:
    print(f"\nGemma 3 initialization failed: {e}\n")
    print("Falling back to non-gated open Qwen 2.5 model (Qwen/Qwen2.5-7B-Instruct)...\n")
    llm = LLM(model="Qwen/Qwen2.5-7B-Instruct", trust_remote_code=True, tensor_parallel_size=1)

sampling_params = SamplingParams(temperature=0.1, max_tokens=512)

In [ ]:
# Test our core Risk Analysis Prompt
system_prompt = "You are an elite payment security AI running on AMD hardware. Evaluate risk."
user_prompt = """
Evaluate the following transaction:
Amount: 15000 PKR
Merchant: Unknown Electronics Store
Device: New Device (Unrecognized)
Location: IP mismatch with billing zip
"""

formatted_prompt = f"<|system|>{system_prompt}<|user|>{user_prompt}<|assistant|>"
outputs = llm.generate([formatted_prompt], sampling_params)

for output in outputs:
    print(f"\n--- AI RISK ANALYSIS ---\n{output.outputs[0].text}")